# Veritas-R1 — Stage 1 temperature scaling (standalone)

Reproduces the temperature-scaling step from `veritas-stage1-eval.ipynb` as a **standalone, runnable notebook** so you can re-fit T on a different validation split (or on a new pull of val data after a deploy) without re-running the full eval.

**Algorithm**: minimise NLL on val logits w.r.t. a single scalar T via L-BFGS. Convex; converges in <100 iterations.

**Reference**: Guo et al. 2017 §4.2.

In [ ]:
%pip install --quiet --upgrade "scipy>=1.13" "matplotlib>=3.8"

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from scipy.special import softmax as np_softmax
import matplotlib.pyplot as plt

ADAPTER_ROOT = Path("/kaggle/working/adapters")
RESULTS_DIR = Path("/kaggle/working/temperature_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

seed_dirs = sorted(d for d in ADAPTER_ROOT.glob("seed_*") if (d / "logits.npz").is_file())
print(f"found {len(seed_dirs)} seeds")

In [ ]:
def fit_temperature(logits, labels, max_iter=200):
    L = torch.as_tensor(logits, dtype=torch.float32)
    Y = torch.as_tensor(labels, dtype=torch.long)
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.01, max_iter=max_iter, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(L / torch.exp(log_T), Y); loss.backward(); return loss
    opt.step(closure)
    return float(np.clip(float(torch.exp(log_T).item()), 0.1, 100.0))

def ece(probs, labels, n_bins=15):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1); correct = (pred == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); val = 0.0; n = len(labels)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == n_bins - 1 else (conf >= lo) & (conf < hi)
        if not mask.any(): continue
        val += mask.sum() / n * abs(conf[mask].mean() - correct[mask].mean())
    return float(val)

def nll(logits, labels, T=1.0):
    return float(F.cross_entropy(torch.as_tensor(logits / T, dtype=torch.float32), torch.as_tensor(labels, dtype=torch.long)).item())

## Fit T per seed and save

In [ ]:
report = []
for d in seed_dirs:
    data = np.load(d / "logits.npz")
    val_logits, val_labels = data["val_logits"], data["val_labels"]
    test_logits, test_labels = data["test_logits"], data["test_labels"]
    T = fit_temperature(val_logits, val_labels)
    raw_probs = np_softmax(test_logits, axis=-1)
    scaled_probs = np_softmax(test_logits / T, axis=-1)
    row = {
        "seed": d.name,
        "T": round(T, 4),
        "val_nll_T_eq_1": round(nll(val_logits, val_labels, 1.0), 4),
        "val_nll_T_fitted": round(nll(val_logits, val_labels, T), 4),
        "test_ece_raw": round(ece(raw_probs, test_labels), 4),
        "test_ece_scaled": round(ece(scaled_probs, test_labels), 4),
        "test_avg_conf_raw": round(float(raw_probs.max(axis=1).mean()), 4),
        "test_avg_conf_scaled": round(float(scaled_probs.max(axis=1).mean()), 4),
    }
    report.append(row)
    (d / "temperature.json").write_text(json.dumps({"T": T}, indent=2))
    print(d.name, row)
(RESULTS_DIR / "temperature_report.json").write_text(json.dumps(report, indent=2))

## Visualise T's effect on confidence distribution

In [ ]:
fig, axes = plt.subplots(1, len(seed_dirs), figsize=(5 * len(seed_dirs), 4), squeeze=False)
for idx, d in enumerate(seed_dirs):
    data = np.load(d / "logits.npz")
    test_logits, test_labels = data["test_logits"], data["test_labels"]
    T = json.loads((d / "temperature.json").read_text())["T"]
    raw_conf = np_softmax(test_logits, axis=-1).max(axis=1)
    scaled_conf = np_softmax(test_logits / T, axis=-1).max(axis=1)
    ax = axes[0, idx]
    ax.hist(raw_conf, bins=30, alpha=0.55, label="raw", color="#a04040")
    ax.hist(scaled_conf, bins=30, alpha=0.55, label=f"T-scaled (T={T:.2f})", color="#3060a0")
    ax.set_xlabel("max-class confidence"); ax.set_ylabel("count"); ax.set_title(d.name); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "confidence_histograms.png", dpi=140); plt.show()